In [8]:
import os
os.getcwd()

'd:\\GitHub\\Sinusoidal-Distribution\\python'

In [13]:
%run sinu.ipynb

In [15]:
import torch
import torch.nn as nn
from torch.autograd import Function

class SinusoidalCDF(Function):
    @staticmethod
    def forward(ctx, z, s, k):
        # Clamp input to support [0, 1] as it's a standard distribution
        z_clamped = torch.clamp(z, 0.001, 0.999)
        
        # Numerical integration for the forward pass (F(z))
        # In a production environment, use a pre-computed look-up table or 
        # a fast hypergeometric series expansion.
        steps = 100
        t = torch.linspace(0, 1, steps).to(z.device)
        # Broadcasting to compute the integral for all elements in z
        # C is the normalization constant we derived earlier
        def get_pdf(val, s_val, k_val):
            return torch.sin(torch.pi * val**s_val)**k_val

        # Simple trapezoidal rule approximation for C and F(z)
        dt = 1.0 / (steps - 1)
        pdf_grid = get_pdf(t, s, k)
        C = torch.sum(pdf_grid) * dt
        
        # Compute CDF for each input z
        # This is a simplified approximation for the forward demonstration
        cdf_z = torch.zeros_like(z_clamped)
        for i, val in enumerate(z_clamped):
            t_sub = torch.linspace(0, val.item(), steps).to(z.device)
            cdf_z[i] = torch.sum(get_pdf(t_sub, s, k)) * (val / (steps - 1))
        
        cdf_z = cdf_z / C
        ctx.save_for_backward(z, s, k, C)
        return cdf_z

    @staticmethod
    def backward(ctx, grad_output):
        z, s, k, C = ctx.saved_tensors
        
        # Gradient wrt z is the PDF (fundamental theorem of calculus)
        dz = (1/C) * torch.sin(torch.pi * torch.pow(z, s))**k
        
        # Gradients wrt s and k would involve differentiating the integral.
        # For a first-order approximation, we use finite differences or 
        # partial derivatives of the integrand.
        ds = torch.zeros_like(s) # Placeholder for complex dF/ds
        dk = torch.zeros_like(k) # Placeholder for complex dF/dk
        
        return grad_output * dz, grad_output.sum() * ds, grad_output.sum() * dk

class SinusoidalNet(nn.Module):
    def __init__(self, in_features, out_features, init_s=2.0, init_k=2.0):
        super(SinusoidalNet, self).__init__()
        self.linear = nn.Linear(in_features, out_features)
        
        # Learnable parameters for the activation function
        self.s = nn.Parameter(torch.tensor([init_s]))
        self.k = nn.Parameter(torch.tensor([init_k]))

    def forward(self, x):
        weighted_sum = self.linear(x)
        # Apply the sinusoidal CDF activation
        return SinusoidalCDF.apply(weighted_sum, self.s, self.k)

In [16]:

# Example Usage:
model = SinusoidalNet(in_features=5, out_features=1)
criterion = nn.MSELoss() # Or your custom DPD Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)